# EXERCISE 01 - COTAÇÃO DO BITCOIN HOJE

**IMPORTS**

In [1]:
import requests
from bs4 import BeautifulSoup
from openai import OpenAI

**CONFIG DO CLIENTE**

In [2]:
Agent = OpenAI(
    base_url= "http://localhost:11434/v1",
    api_key= "ollama"
)

**HEADERS**

In [3]:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"   
}

**CLASSE DO SITE**

In [4]:
class Website:
    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, "html.parser")
        self.title = soup.title.string if soup.title else "No title found"

        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()

        self.text = soup.body.get_text(separator="\n", strip=True)

**FUNÇÃO PARA FILTRAR A MOEDA**

In [5]:
def filtrar_moedas(texto):
    palavras_chave = [
        "bitcoin", "btc"
    ]

    linhas = texto.splitlines()

    linhas_filtradas = [
        linha.strip()
        for linha in linhas
        if linha.strip() and any(p in linha.lower() for p in palavras_chave)
    ]

    return "\n".join(linhas_filtradas)

**CAPTURAR O SITE**

In [6]:
cotacao = Website("https://bitsgap.com/pt/converter/btc/usd")

**FILTRAR O TEXTO**

In [7]:
texto_filtrado = filtrar_moedas(cotacao.text)
print(texto_filtrado)

BTC
Conversor de Preço em Tempo Real de BTC para USD
BTC
Bitcoin
Calculadora de BTC para USD
BTC
0.0000147 BTC
Negocie BTC
O Bitcoin (BTC), fundado em 2009 por um criador anônimo com o pseudônimo "Satoshi Nakamoto", é a primeira criptomoeda descentralizada e sistema de pagamento a alcançar ampla adoção.
Embora o conceito de moedas digitais não seja novo, o Bitcoin é único porque não é controlado por nenhuma autoridade central. Em vez de depender da infraestrutura financeira convencional, as transações em Bitcoin ocorrem diretamente entre os usuários no livro-razão distribuído conhecido como blockchain. A blockchain permite que as transações sejam verificadas, registradas e organizadas de forma transparente e imutável. Quando novas transações são aprovadas e adicionadas ao livro-razão, a rede atualiza a cópia do livro-razão de cada usuário para mostrar as alterações mais recentes.
Devido à sua natureza descentralizada, o Bitcoin não tem uma fonte única de falha, o que torna quase imposs

**MONTAR O PROMPT**

In [8]:
def user_prompt_for(website, texto_filtrado):
    prompt = f"Você está olhando para um site com o título {website.title}\n"
    prompt += "O conteúdo relevante sobre moedas é o seguinte:\n\n"
    prompt += texto_filtrado
    return prompt

**GERAR O PROMPT FINAL**

In [9]:
prompt_usuario = user_prompt_for(cotacao, texto_filtrado)
print(prompt_usuario)

Você está olhando para um site com o título Conversor de BTC/USD da Bitsgap: transforme Bitcoin em dólar americano | Bitsgap
O conteúdo relevante sobre moedas é o seguinte:

BTC
Conversor de Preço em Tempo Real de BTC para USD
BTC
Bitcoin
Calculadora de BTC para USD
BTC
0.0000147 BTC
Negocie BTC
O Bitcoin (BTC), fundado em 2009 por um criador anônimo com o pseudônimo "Satoshi Nakamoto", é a primeira criptomoeda descentralizada e sistema de pagamento a alcançar ampla adoção.
Embora o conceito de moedas digitais não seja novo, o Bitcoin é único porque não é controlado por nenhuma autoridade central. Em vez de depender da infraestrutura financeira convencional, as transações em Bitcoin ocorrem diretamente entre os usuários no livro-razão distribuído conhecido como blockchain. A blockchain permite que as transações sejam verificadas, registradas e organizadas de forma transparente e imutável. Quando novas transações são aprovadas e adicionadas ao livro-razão, a rede atualiza a cópia do liv

**ENVIAR O MODELO**

In [10]:
response = Agent.chat.completions.create(
    model="llama3.2",
    messages=[
        {"role": "system", "content": "Você é um assistente financeiro que resume cotações de bitcoin de forma curta e objetiva."},
        {"role": "user", "content": prompt_usuario}
    ]
)

print(response.choices[0].message.content)

Aqui está uma resumo do conteúdo relevante sobre o Bitcoin:

**Preço atual:** $68,217.02 USD (com uma variação de 0.15% nos últimos 24 horas e -3.89% nos últimos 7 dias)

**Capitalização de mercado em tempo real:** $1.4 trilhões

**Volume de negociação das últimas 24 horas:** $37 bi

**Conversor de BTC para USD:** O valor atual de 1 BTC é $68,217.02 USD e o valor anterior do mês passado era $65,847.65 USD, uma queda de 3.47% em relação ao preço atual.

Essas são as informações mais recentes sobre o preço do Bitcoin disponíveis no site da Bitsgap.
